# Source Scope — forecaster backtest

> **The data is synthetic.** Every price here comes from a seeded generator, not a market.
> See [`docs/data-sources.md`](../../docs/data-sources.md). The methodology transfers; the numbers do not.

Three one-step-ahead forecasters compared on a **rolling-origin walk-forward split**: train on weeks `0..t`, predict week `t+1`, advance `t`. Nothing is shuffled, and no model ever sees a week at or beyond the one it is predicting.

| Model | What it is |
|---|---|
| `naive` | Last observed weekly median. The baseline that must be beaten. |
| `sarima` | SARIMAX on the log median series, order chosen by AIC. |
| `gbm` | Gradient-boosted trees on lag/rolling/dispersion features, trained across all parts. |

Run `python scripts/backtest.py` first — this notebook reads what it wrote.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / 'ml-service'))
OUT = REPO_ROOT / 'ml-service' / 'data' / 'out'

predictions = pd.read_csv(OUT / 'predictions.csv')
summary = json.loads((OUT / 'backtest_summary.json').read_text())

print(f"{summary['parts']} parts, {summary['predictions']:,} predictions, "
      f"{summary['elapsed_seconds']:.0f}s")
predictions.head()

## Error by model and segment

MAPE is the headline because it is scale-free — this catalog spans $0.01 resistors to $800 FPGAs, and an unnormalised error would simply rank the models by how well they do on the expensive parts. RMSE is reported alongside it for exactly that reason: read it as a dollar figure dominated by the top of the price range, not as a second opinion on accuracy.

In [ ]:
def metrics(frame):
    ape = (frame.actual - frame.predicted).abs() / frame.actual
    return pd.Series({
        'n': len(frame),
        'MAPE': ape.mean(),
        'median APE': ape.median(),
        'RMSE': np.sqrt(((frame.actual - frame.predicted) ** 2).mean()),
    })


predictions['segment_lifecycle'] = np.where(
    predictions.lifecycle.isin(['Obsolete', 'EOL']), 'obsolete', 'active')
predictions['segment_market'] = np.where(
    predictions.during_spike == 1, 'during_spike', 'normal')

overall = predictions.groupby('model').apply(metrics, include_groups=False)
display(overall.style.format({'n': '{:,.0f}', 'MAPE': '{:.1%}',
                              'median APE': '{:.1%}', 'RMSE': '{:.4f}'}))

In [ ]:
by_lifecycle = (predictions.groupby(['segment_lifecycle', 'model'])
                .apply(metrics, include_groups=False))
by_market = (predictions.groupby(['segment_market', 'model'])
             .apply(metrics, include_groups=False))

for title, table in [('lifecycle', by_lifecycle), ('market condition', by_market)]:
    print(f'\n=== by {title} ===')
    display(table.style.format({'n': '{:,.0f}', 'MAPE': '{:.1%}',
                                'median APE': '{:.1%}', 'RMSE': '{:.4f}'}))

## Which model is closest, per part-week

Mean error can hide a model that is usually mediocre but occasionally excellent. Counting outright wins shows whether a model is broadly useful or just avoids catastrophe.

In [ ]:
wide = predictions.pivot_table(index=['mpn', 'week_index'],
                               columns='model', values='predicted')
actual = predictions.pivot_table(index=['mpn', 'week_index'],
                                 columns='model', values='actual').iloc[:, 0]
errors = wide.sub(actual, axis=0).abs().div(actual, axis=0)
winners = errors.idxmin(axis=1).value_counts(normalize=True)

fig, ax = plt.subplots(figsize=(7, 3))
winners.sort_values().plot.barh(ax=ax, color='#4a6fa5')
ax.set_xlabel('share of part-weeks where this model was closest')
ax.xaxis.set_major_formatter(lambda x, _: f'{x:.0%}')
ax.set_ylabel('')
ax.set_title('Closest forecast, counted per part-week')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()
winners

## Actual versus predicted

Four parts: the two whose shortage spike was sharpest, and two ordinary ones. The spiked pair is where the models actually differ — on a calm part all three lines sit on top of each other, which is itself the finding.

In [ ]:
spread = (predictions[predictions.model == 'naive']
          .groupby('mpn')
          .apply(lambda f: f.actual.max() / max(f.actual.min(), 1e-9),
                 include_groups=False)
          .sort_values(ascending=False))

examples = list(spread.head(2).index) + list(spread.tail(2).index)
colours = {'naive': '#999999', 'sarima': '#c0603a', 'gbm': '#4a6fa5'}

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for ax, mpn in zip(axes.ravel(), examples):
    part = predictions[predictions.mpn == mpn]
    truth = part[part.model == 'naive'].sort_values('week_index')
    ax.plot(truth.week_index, truth.actual, color='#111111', lw=2.2,
            label='actual', zorder=5)
    for model, colour in colours.items():
        rows = part[part.model == model].sort_values('week_index')
        ax.plot(rows.week_index, rows.predicted, color=colour, lw=1.3,
                ls='--', label=model)
    swing = truth.actual.max() / max(truth.actual.min(), 1e-9)
    ax.set_title(f'{mpn}  ({part.lifecycle.iloc[0]}, {swing:.1f}x range)', fontsize=10)
    ax.set_ylabel('USD')
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
axes[0][0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel('week index')
plt.tight_layout()
plt.show()

## Error through a shortage

Averaged across every part, aligned on the weeks the heat index flagged as a spike. This is the plot that says whether the extra machinery is worth anything: if the models only separate here, then that is the only place they should be trusted to.

In [ ]:
predictions['ape'] = ((predictions.actual - predictions.predicted).abs()
                      / predictions.actual)

fig, ax = plt.subplots(figsize=(9, 4))
for model, colour in colours.items():
    rows = predictions[predictions.model == model]
    curve = rows.groupby('segment_market').ape.mean()
    ax.bar([f'{model}\n{seg}' for seg in curve.index], curve.values,
           color=colour, width=0.6)
ax.set_ylabel('MAPE')
ax.yaxis.set_major_formatter(lambda y, _: f'{y:.0%}')
ax.set_title('Error in normal weeks versus shortage weeks')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

print('\n'.join(summary['observations']))

## Read the write-up

The commentary — including where the naive baseline wins and why that is the expected result rather than an embarrassment — is in [`docs/backtest-results.md`](../../docs/backtest-results.md).